# Delivery Performance, Delay Risk & Logistics Efficiency Analysis

This notebook performs end-to-end exploratory data analysis on the APL Logistics dataset and reproduces the core metrics used by the Streamlit dashboard.

**Dataset limitation:** the supplied CSV has no order/shipping date field, so time-based trend analysis cannot be performed on this version.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "data/APL_Logistics.csv"
df = pd.read_csv(DATA_PATH, encoding="latin1")
df.shape


In [ ]:
df.info()


In [ ]:
# Derived fields
df["Delivery Gap (Days)"] = df["Days for shipping (real)"] - df["Days for shipment (scheduled)"]
df["On Time Flag"] = (df["Late_delivery_risk"] == 0).astype(int)

df[["Days for shipping (real)", "Days for shipment (scheduled)", "Delivery Gap (Days)", "Late_delivery_risk"]].head()


## 1. Key Performance Indicators

In [ ]:
kpis = {
    "Total Orders": len(df),
    "On-Time Delivery Rate (%)": df["On Time Flag"].mean() * 100,
    "Average Delivery Gap (Days)": df["Delivery Gap (Days)"].mean(),
    "Late Delivery Risk Ratio (%)": df["Late_delivery_risk"].mean() * 100,
    "Average Delay Among Late Orders (Days)": df.loc[df["Delivery Gap (Days)"] > 0, "Delivery Gap (Days)"].mean(),
}
pd.Series(kpis).round(2)


## 2. Delivery Performance Overview

In [ ]:
delivery_status = df["Delivery Status"].value_counts()
delivery_status


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(delivery_status.index, delivery_status.values)
ax.set_title("Delivery Status Distribution")
ax.set_ylabel("Orders")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


## 3. Delay Risk Analysis

In [ ]:
risk_distribution = df["Late_delivery_risk"].value_counts().sort_index()
risk_distribution.index = ["No late risk", "Late risk"]
risk_distribution


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(risk_distribution.index, risk_distribution.values)
ax.set_title("Late Delivery Risk Distribution")
ax.set_ylabel("Orders")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["Delivery Gap (Days)"], bins=10)
ax.set_title("Delivery Gap Histogram")
ax.set_xlabel("Actual Shipping Days - Scheduled Shipping Days")
ax.set_ylabel("Orders")
plt.tight_layout()
plt.show()


## 4. Shipping Mode Efficiency Analysis

In [ ]:
shipping_mode = (
    df.groupby("Shipping Mode")
      .agg(
          Orders=("Late_delivery_risk", "size"),
          Late_Risk_Ratio=("Late_delivery_risk", "mean"),
          Avg_Delivery_Gap=("Delivery Gap (Days)", "mean"),
          SLA_Compliance=("On Time Flag", "mean"),
      )
      .reset_index()
)
shipping_mode["Late_Risk_Ratio"] *= 100
shipping_mode["SLA_Compliance"] *= 100
shipping_mode["Shipping_Mode_Efficiency_Index"] = shipping_mode["SLA_Compliance"]
shipping_mode.round(2)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = shipping_mode.sort_values("SLA_Compliance", ascending=False)
ax.bar(plot_df["Shipping Mode"], plot_df["SLA_Compliance"])
ax.set_title("SLA Compliance by Shipping Mode")
ax.set_ylabel("SLA Compliance (%)")
plt.tight_layout()
plt.show()


## 5. Regional & Market Diagnostics

In [ ]:
regional = (
    df.groupby(["Market", "Order Region"])
      .agg(
          Orders=("Late_delivery_risk", "size"),
          Late_Risk_Ratio=("Late_delivery_risk", "mean"),
          Avg_Delivery_Gap=("Delivery Gap (Days)", "mean"),
      )
      .reset_index()
)
regional["Late_Risk_Ratio"] *= 100
regional.sort_values("Late_Risk_Ratio", ascending=False).head(15).round(2)


In [ ]:
region_rank = (
    df.groupby("Order Region")
      .agg(
          Orders=("Late_delivery_risk", "size"),
          Late_Risk_Ratio=("Late_delivery_risk", "mean"),
          Avg_Delivery_Gap=("Delivery Gap (Days)", "mean"),
      )
      .reset_index()
)
region_rank["Late_Risk_Ratio"] *= 100
region_rank.sort_values("Late_Risk_Ratio", ascending=False).head(10).round(2)


## 6. Customer Segment Impact Analysis

In [ ]:
segment = (
    df.groupby("Customer Segment")
      .agg(
          Orders=("Late_delivery_risk", "size"),
          Late_Risk_Ratio=("Late_delivery_risk", "mean"),
          Avg_Delivery_Gap=("Delivery Gap (Days)", "mean"),
      )
      .reset_index()
)
segment["Late_Risk_Ratio"] *= 100
segment.round(2)


## 7. Main Insights

- Overall late-delivery risk is high relative to on-time performance.
- Standard Class is the strongest shipping mode by SLA compliance in this dataset.
- First Class and Second Class require the most attention because their late-risk ratios are substantially higher.
- Regional risk is distributed across multiple markets/regions, supporting region-specific scorecards.
- Customer-segment differences are small compared with shipping-mode differences, suggesting that operational routing and SLA execution are the more important intervention areas.


## 8. Recommendations

1. Review SLA design and operational execution for First Class and Second Class.
2. Create automated alerts when delivery gap becomes positive or late-risk flags increase.
3. Use Standard Class performance as a benchmark for process stability.
4. Maintain a regional late-risk scorecard to identify emerging geographic hotspots.
5. Prioritize high-value or SLA-sensitive orders for more reliable routes when operationally feasible.
6. Add order/shipping timestamps to future datasets to enable trend analysis and date-range filtering.
